# GoEmotions 6-Class RoBERTa Full Pipeline
This notebook runs an end-to-end pipeline in an isolated experiment folder:
1. Merge raw GoEmotions CSVs
2. Clean + map to 6 classes
3. Stratified train/valid/test split
4. Fine-tune `roberta-base` with weighted loss, early stopping, ReduceLROnPlateau, and fallbacks
5. Export full evaluation artifacts


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
%pip -q install transformers datasets evaluate accelerate scikit-learn pandas numpy matplotlib seaborn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.7 MB/s eta 0:00:00


In [5]:
import json
import os
import random
import re
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import evaluate
import seaborn as sns
import matplotlib.pyplot as plt

from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)
from sklearn.utils.class_weight import compute_class_weight

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    TrainerCallback,
    set_seed,
)

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# The find_repo_root function is not suitable for a Colab environment where the project root
# is located on Google Drive and Path.cwd() is not a subdirectory of the drive.
# We will directly set REPO_ROOT to the absolute path of the project's base directory.
# def find_repo_root(start: Path) -> Path:
#     start = start.resolve()
#     for p in [start] + list(start.parents):
#         if (p / "data" / "raw" / "goemotions").exists():
#             return p
#     raise FileNotFoundError("Could not find repo root containing /content/drive/MyDrive/Emotional Aware Chatbot/data/raw/goemotions")

REPO_ROOT = Path("/content/drive/MyDrive/Emotional Aware Chatbot")
EXP_ROOT = REPO_ROOT / "experiments" / "goemotions_roberta_full_pipeline"
NOTEBOOK_DIR = EXP_ROOT / "notebooks"
DATA_INTERIM_DIR = EXP_ROOT / "data" / "interim"
DATA_PROCESSED_DIR = EXP_ROOT / "data" / "processed"
ARTIFACTS_DIR = EXP_ROOT / "artifacts"
MODELS_DIR = EXP_ROOT / "models" / "best_model"
RESULTS_DIR = EXP_ROOT / "results" / "trainer_runs"
REPORTS_DIR = EXP_ROOT / "reports"

for d in [
    NOTEBOOK_DIR,
    DATA_INTERIM_DIR,
    DATA_PROCESSED_DIR,
    ARTIFACTS_DIR,
    MODELS_DIR,
    RESULTS_DIR,
    REPORTS_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)

RAW_FILES = [
    REPO_ROOT / "data" / "raw" / "goemotions" / "goemotions_1.csv",
    REPO_ROOT / "data" / "raw" / "goemotions" / "goemotions_2.csv",
    REPO_ROOT / "data" / "raw" / "goemotions" / "goemotions_3.csv",
]

for f in RAW_FILES:
    if not f.exists():
        raise FileNotFoundError(f"Missing raw file: {f}")

if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

print("Repo root:", REPO_ROOT)
print("Experiment root:", EXP_ROOT)
print("Device:", DEVICE)


Repo root: /content/drive/MyDrive/Emotional Aware Chatbot
Experiment root: /content/drive/MyDrive/Emotional Aware Chatbot/experiments/goemotions_roberta_full_pipeline
Device: cuda


In [6]:
# Load and merge raw files
raw_dfs = [pd.read_csv(p) for p in RAW_FILES]
raw_merged = pd.concat(raw_dfs, axis=0, ignore_index=True)

merged_path = DATA_INTERIM_DIR / "merged_raw.csv"
raw_merged.to_csv(merged_path, index=False)

print("Merged shape:", raw_merged.shape)
print("Saved merged raw to:", merged_path)


Merged shape: (211225, 37)
Saved merged raw to: /content/drive/MyDrive/Emotional Aware Chatbot/experiments/goemotions_roberta_full_pipeline/data/interim/merged_raw.csv


In [7]:
# Detect valid binary label columns and define mapping
META_COLS = {
    "text", "id", "author", "subreddit", "link_id", "parent_id", "created_utc", "rater_id"
}

candidate_cols = [c for c in raw_merged.columns if c not in META_COLS]
binary_label_cols = []
for c in candidate_cols:
    vals = set(raw_merged[c].dropna().unique().tolist())
    if vals.issubset({0, 1}):
        binary_label_cols.append(c)

print("Detected binary label columns:", len(binary_label_cols))
print(sorted(binary_label_cols))

EMO_MAP = {
    "Anger": ["anger", "annoyance", "disapproval", "disgust"],
    "Fear": ["fear", "nervousness"],
    "Joy": ["joy", "amusement", "excitement", "gratitude", "optimism", "pride", "relief"],
    "Love": ["love", "admiration", "caring", "desire"],
    "Sadness": ["sadness", "grief", "disappointment", "remorse", "embarrassment"],
    "Neutral": ["neutral"],
}

EXPECTED_CLASSES = ["Anger", "Fear", "Joy", "Love", "Neutral", "Sadness"]
assert sorted(EMO_MAP.keys()) == sorted(EXPECTED_CLASSES)

mapping_rows = []
for final_class, fine_labels in EMO_MAP.items():
    for fine in fine_labels:
        mapping_rows.append({"fine_label": fine, "final_class": final_class})

mapping_df = pd.DataFrame(mapping_rows).sort_values(["final_class", "fine_label"]).reset_index(drop=True)
mapping_df.to_csv(ARTIFACTS_DIR / "label_mapping.csv", index=False)
print("Saved mapping to:", ARTIFACTS_DIR / "label_mapping.csv")


Detected binary label columns: 29
['admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval', 'disgust', 'embarrassment', 'example_very_unclear', 'excitement', 'fear', 'gratitude', 'grief', 'joy', 'love', 'nervousness', 'neutral', 'optimism', 'pride', 'realization', 'relief', 'remorse', 'sadness', 'surprise']
Saved mapping to: /content/drive/MyDrive/Emotional Aware Chatbot/experiments/goemotions_roberta_full_pipeline/artifacts/label_mapping.csv


In [8]:
# Map each row to a single final class (drop ambiguous/unmapped rows)
def clean_text(text: str) -> str:
    text = str(text).strip()
    text = re.sub(r"http\S+|www\.\S+", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


fine_to_group = {}
for group, fine_labels in EMO_MAP.items():
    for fine in fine_labels:
        fine_to_group[fine] = group


def row_to_final_label(row: pd.Series):
    active = [c for c in binary_label_cols if row.get(c, 0) == 1]
    if not active:
        return None, "drop_no_active"

    votes = {k: 0 for k in EMO_MAP.keys()}
    for lab in active:
        if lab in fine_to_group:
            votes[fine_to_group[lab]] += 1

    # only neutral active
    if votes["Neutral"] > 0 and sum(votes.values()) == votes["Neutral"]:
        return "Neutral", "keep_neutral_only"

    non_neutral_votes = {k: v for k, v in votes.items() if k != "Neutral"}
    max_vote = max(non_neutral_votes.values())

    if max_vote == 0:
        return None, "drop_unmapped"

    winners = [k for k, v in non_neutral_votes.items() if v == max_vote]
    if len(winners) > 1:
        return None, "drop_ambiguous"

    return winners[0], "keep_non_neutral"


work_df = raw_merged.copy()
work_df["text"] = work_df["text"].astype(str).apply(clean_text)

labels_and_reasons = work_df.apply(row_to_final_label, axis=1)
work_df["final_emotion"] = labels_and_reasons.apply(lambda x: x[0])
work_df["filter_reason"] = labels_and_reasons.apply(lambda x: x[1])

stats = (
    work_df["filter_reason"]
    .value_counts(dropna=False)
    .rename_axis("reason")
    .reset_index(name="count")
)
stats["ratio"] = (stats["count"] / len(work_df)).round(6)
stats.to_csv(ARTIFACTS_DIR / "class_distribution.csv", index=False)

print("Filter stats:")
print(stats)
print("Saved stats to:", ARTIFACTS_DIR / "class_distribution.csv")


Filter stats:
              reason   count     ratio
0   keep_non_neutral  107803  0.510370
1  keep_neutral_only   55298  0.261797
2      drop_unmapped   36458  0.172603
3     drop_ambiguous   11666  0.055230
Saved stats to: /content/drive/MyDrive/Emotional Aware Chatbot/experiments/goemotions_roberta_full_pipeline/artifacts/class_distribution.csv


In [9]:
# Keep valid mapped rows, remove empties and duplicates, validate label set
final_df = work_df.dropna(subset=["final_emotion"])[["text", "final_emotion"]].copy()
final_df = final_df[final_df["text"].str.len() > 0].copy()
final_df = final_df.drop_duplicates(subset=["text"]).reset_index(drop=True)

label_set = sorted(final_df["final_emotion"].unique().tolist())
if label_set != sorted(EXPECTED_CLASSES):
    raise ValueError(f"Label mismatch. Expected {sorted(EXPECTED_CLASSES)}, got {label_set}")

print("Final cleaned shape:", final_df.shape)
print(final_df["final_emotion"].value_counts(normalize=True).round(4))

# Imbalance check + augmentation-based balancing (before split)
class_counts = final_df["final_emotion"].value_counts()
max_count = int(class_counts.max())
min_count = int(class_counts.min())
imbalance_ratio = max_count / max(min_count, 1)
IMBALANCE_THRESHOLD = 1.5

print(f"Imbalance ratio (max/min): {imbalance_ratio:.2f}")
print("Class counts before balancing:")
print(class_counts)


def augment_text(text: str, rng: random.Random) -> str:
    words = text.split()
    if len(words) == 0:
        return text

    op = rng.choice(["swap", "delete", "duplicate", "punct"])

    if op == "swap" and len(words) >= 2:
        i = rng.randint(0, len(words) - 2)
        words[i], words[i + 1] = words[i + 1], words[i]
        return " ".join(words)

    if op == "delete" and len(words) >= 4:
        i = rng.randint(0, len(words) - 1)
        words = [w for j, w in enumerate(words) if j != i]
        return " ".join(words)

    if op == "duplicate":
        i = rng.randint(0, len(words) - 1)
        words.insert(i, words[i])
        return " ".join(words)

    if op == "punct":
        punct = rng.choice(["!", "...", "?!"])
        return (text + punct).strip()

    return text


if imbalance_ratio > IMBALANCE_THRESHOLD:
    rng = random.Random(SEED)
    existing_texts = set(final_df["text"].tolist())
    aug_rows = []

    for cls in EXPECTED_CLASSES:
        cls_df = final_df[final_df["final_emotion"] == cls]
        need = max_count - len(cls_df)
        if need <= 0:
            continue

        base_texts = cls_df["text"].tolist()
        generated = 0
        attempts = 0
        max_attempts = need * 20

        while generated < need and attempts < max_attempts:
            attempts += 1
            base = rng.choice(base_texts)
            aug = augment_text(base, rng)

            if not aug or len(aug.strip()) == 0:
                continue
            if aug in existing_texts:
                continue

            existing_texts.add(aug)
            aug_rows.append({"text": aug, "final_emotion": cls, "is_augmented": 1})
            generated += 1

        if generated < need:
            # Controlled fallback to guarantee exact balance if augmentation collisions are high.
            for i in range(need - generated):
                base = rng.choice(base_texts)
                aug = f"{base} [AUG_{cls}_{i}]"
                if aug in existing_texts:
                    continue
                existing_texts.add(aug)
                aug_rows.append({"text": aug, "final_emotion": cls, "is_augmented": 1})

    balanced_df = final_df.copy()
    balanced_df["is_augmented"] = 0
    if len(aug_rows) > 0:
        balanced_df = pd.concat([balanced_df, pd.DataFrame(aug_rows)], ignore_index=True)

    balanced_df = balanced_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
else:
    balanced_df = final_df.copy()
    balanced_df["is_augmented"] = 0

print("Balanced dataset shape:", balanced_df.shape)
print("Class counts after balancing:")
print(balanced_df["final_emotion"].value_counts())

augmented_count = int((balanced_df["is_augmented"] == 1).sum())
original_count = int((balanced_df["is_augmented"] == 0).sum())
print("Augmentation applied:", augmented_count > 0)
print(f"Original rows: {original_count}, Augmented rows: {augmented_count}")

if augmented_count > 0:
    print("Augmented rows by class:")
    print(
        balanced_df[balanced_df["is_augmented"] == 1]["final_emotion"]
        .value_counts()
        .reindex(EXPECTED_CLASSES)
        .fillna(0)
        .astype(int)
    )

balance_stats = (
    balanced_df.groupby(["final_emotion", "is_augmented"]).size().reset_index(name="count")
)
balance_stats.to_csv(ARTIFACTS_DIR / "balanced_class_distribution.csv", index=False)
balanced_df[["text", "final_emotion", "is_augmented"]].to_csv(
    DATA_INTERIM_DIR / "balanced_dataset_with_flags.csv", index=False
)
print("Saved balanced stats to:", ARTIFACTS_DIR / "balanced_class_distribution.csv")
print("Saved balanced dataset with flags to:", DATA_INTERIM_DIR / "balanced_dataset_with_flags.csv")


Final cleaned shape: (55633, 2)
final_emotion
Neutral    0.3560
Joy        0.2101
Anger      0.1716
Love       0.1586
Sadness    0.0841
Fear       0.0196
Name: proportion, dtype: float64
Imbalance ratio (max/min): 18.14
Class counts before balancing:
final_emotion
Neutral    19808
Joy        11689
Anger       9546
Love        8822
Sadness     4676
Fear        1092
Name: count, dtype: int64
Balanced dataset shape: (118848, 3)
Class counts after balancing:
final_emotion
Neutral    19808
Sadness    19808
Fear       19808
Love       19808
Anger      19808
Joy        19808
Name: count, dtype: int64
Augmentation applied: True
Original rows: 55633, Augmented rows: 63215
Augmented rows by class:
final_emotion
Anger      10262
Fear       18716
Joy         8119
Love       10986
Neutral        0
Sadness    15132
Name: count, dtype: int64
Saved balanced stats to: /content/drive/MyDrive/Emotional Aware Chatbot/experiments/goemotions_roberta_full_pipeline/artifacts/balanced_class_distribution.csv
Sa

In [10]:
# Stratified 80/10/10 split (from balanced data)
model_df = balanced_df[["text", "final_emotion"]].copy()

train_df, temp_df = train_test_split(
    model_df,
    test_size=0.2,
    random_state=SEED,
    stratify=model_df["final_emotion"],
)

valid_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=SEED,
    stratify=temp_df["final_emotion"],
)

for name, df in [("train", train_df), ("valid", valid_df), ("test", test_df)]:
    out = DATA_PROCESSED_DIR / f"{name}.csv"
    df.to_csv(out, index=False)
    print(name, df.shape, "->", out)

# Data integrity checks
assert train_df["text"].isna().sum() == 0
assert valid_df["text"].isna().sum() == 0
assert test_df["text"].isna().sum() == 0
assert (train_df["text"].str.len() > 0).all()
assert (valid_df["text"].str.len() > 0).all()
assert (test_df["text"].str.len() > 0).all()
assert set(train_df["final_emotion"].unique()) == set(EXPECTED_CLASSES)
assert set(valid_df["final_emotion"].unique()) == set(EXPECTED_CLASSES)
assert set(test_df["final_emotion"].unique()) == set(EXPECTED_CLASSES)
assert len(train_df) + len(valid_df) + len(test_df) == len(model_df)

# Stratification quality check
full_dist = model_df["final_emotion"].value_counts(normalize=True)
for split_name, split_df in [("train", train_df), ("valid", valid_df), ("test", test_df)]:
    split_dist = split_df["final_emotion"].value_counts(normalize=True)
    aligned = pd.concat([full_dist, split_dist], axis=1).fillna(0)
    aligned.columns = ["full", "split"]
    max_delta = (aligned["split"] - aligned["full"]).abs().max()
    print(f"Max class proportion delta for {split_name}: {max_delta:.4f}")
    assert max_delta < 0.015, f"Stratification drift too high for {split_name}: {max_delta}"


train (95078, 2) -> /content/drive/MyDrive/Emotional Aware Chatbot/experiments/goemotions_roberta_full_pipeline/data/processed/train.csv
valid (11885, 2) -> /content/drive/MyDrive/Emotional Aware Chatbot/experiments/goemotions_roberta_full_pipeline/data/processed/valid.csv
test (11885, 2) -> /content/drive/MyDrive/Emotional Aware Chatbot/experiments/goemotions_roberta_full_pipeline/data/processed/test.csv
Max class proportion delta for train: 0.0000
Max class proportion delta for valid: 0.0001
Max class proportion delta for test: 0.0001


In [11]:
# Label encoding + HF datasets
labels = sorted(EXPECTED_CLASSES)
label2id = {l: i for i, l in enumerate(labels)}
id2label = {i: l for l, i in label2id.items()}

train_df = train_df.copy()
valid_df = valid_df.copy()
test_df = test_df.copy()

train_df["label"] = train_df["final_emotion"].map(label2id)
valid_df["label"] = valid_df["final_emotion"].map(label2id)
test_df["label"] = test_df["final_emotion"].map(label2id)

train_ds = Dataset.from_pandas(train_df[["text", "label"]], preserve_index=False)
valid_ds = Dataset.from_pandas(valid_df[["text", "label"]], preserve_index=False)
test_ds = Dataset.from_pandas(test_df[["text", "label"]], preserve_index=False)

MODEL_NAME = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=128)

train_tok = train_ds.map(tokenize, batched=True)
valid_tok = valid_ds.map(tokenize, batched=True)
test_tok = test_ds.map(tokenize, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Class weights from train split
classes = np.array(sorted(train_df["label"].unique().tolist()))
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_df["label"].to_numpy(),
)
class_weights = torch.tensor(class_weights, dtype=torch.float)
print("Class weights:", class_weights)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/95078 [00:00<?, ? examples/s]

Map:   0%|          | 0/11885 [00:00<?, ? examples/s]

Map:   0%|          | 0/11885 [00:00<?, ? examples/s]

Class weights: tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])


In [12]:
# Custom trainer + metrics + callbacks
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")


def compute_metrics(eval_pred):
    logits, y_true = eval_pred
    y_pred = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_metric.compute(predictions=y_pred, references=y_true)["accuracy"],
        "f1_macro": f1_metric.compute(predictions=y_pred, references=y_true, average="macro")["f1"],
        "f1_weighted": f1_metric.compute(predictions=y_pred, references=y_true, average="weighted")["f1"],
    }


class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        cw = self.class_weights
        if cw is not None:
            cw = cw.to(logits.device)
        loss_fct = torch.nn.CrossEntropyLoss(weight=cw)
        loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss


class StagnationOrNaNCallback(TrainerCallback):
    def __init__(self, threshold=0.001, patience=2):
        self.threshold = threshold
        self.patience = patience
        self.best = -float("inf")
        self.no_improve = 0
        self.triggered = False
        self.reason = None

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics is None:
            return control

        eval_loss = metrics.get("eval_loss")
        eval_f1 = metrics.get("eval_f1_macro")

        if eval_loss is not None and (np.isnan(eval_loss) or np.isinf(eval_loss)):
            self.triggered = True
            self.reason = "nan_or_inf_loss"
            control.should_training_stop = True
            return control

        if eval_f1 is None:
            return control

        if eval_f1 > self.best + self.threshold:
            self.best = eval_f1
            self.no_improve = 0
        else:
            self.no_improve += 1

        if self.no_improve >= self.patience:
            self.triggered = True
            self.reason = "f1_stagnation"
            control.should_training_stop = True

        return control


In [ ]:
# Train with OOM fallback and one auto-LR fallback retry

# Fast temporary mode for quick iterations on local machine.
FAST_TRAINING = False
FAST_TRAIN_SAMPLES = 12000
FAST_VALID_SAMPLES = 2000
FAST_NUM_EPOCHS = 2
FAST_MAX_STEPS = 600


def clear_memory():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    if torch.backends.mps.is_available():
        try:
            torch.mps.empty_cache()
        except Exception:
            pass


BASE_TRAIN_BS = 8
BASE_EVAL_BS = 8
BASE_GRAD_ACC = 1
BASE_LR = 2e-5

if FAST_TRAINING:
    train_runtime = train_tok.shuffle(seed=SEED).select(range(min(FAST_TRAIN_SAMPLES, len(train_tok))))
    valid_runtime = valid_tok.shuffle(seed=SEED).select(range(min(FAST_VALID_SAMPLES, len(valid_tok))))
    runtime_epochs = FAST_NUM_EPOCHS
    runtime_max_steps = FAST_MAX_STEPS
else:
    train_runtime = train_tok
    valid_runtime = valid_tok
    runtime_epochs = 25   #Earlier 50
    runtime_max_steps = -1

print(
    "Training mode:",
    "FAST" if FAST_TRAINING else "FULL",
    "| train samples:", len(train_runtime),
    "| valid samples:", len(valid_runtime),
    "| epochs:", runtime_epochs,
    "| max_steps:", runtime_max_steps,
)


def build_trainer(train_bs=16, eval_bs=32, grad_acc=1, lr=2e-5):
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(labels),
        id2label=id2label,
        label2id=label2id,
    )

    training_args = TrainingArguments(
        output_dir=str(RESULTS_DIR),
        eval_strategy="steps" if FAST_TRAINING else "epoch",
        eval_steps=100 if FAST_TRAINING else None,
        save_strategy="steps" if FAST_TRAINING else "epoch",
        save_steps=100 if FAST_TRAINING else None,
        logging_strategy="steps",
        logging_steps=50 if FAST_TRAINING else 100,
        learning_rate=lr,
        lr_scheduler_type="reduce_lr_on_plateau",
        per_device_train_batch_size=train_bs,
        per_device_eval_batch_size=eval_bs,
        gradient_accumulation_steps=grad_acc,
        num_train_epochs=runtime_epochs,
        max_steps=runtime_max_steps,
        weight_decay=0.01,
        max_grad_norm=1.0,
        warmup_ratio=0.06,
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True,
        save_total_limit=2,
        report_to="none",
        dataloader_num_workers=2,
        seed=SEED,
    )

    stagnation_cb = StagnationOrNaNCallback(threshold=0.001, patience=2)

    trainer = WeightedTrainer(
        model=model,
        args=training_args,
        train_dataset=train_runtime,
        eval_dataset=valid_runtime,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        class_weights=class_weights,
        callbacks=[
            EarlyStoppingCallback(early_stopping_patience=2, early_stopping_threshold=0.001),
            stagnation_cb,
        ],
    )
    return trainer, stagnation_cb


trainer, stagnation_cb = build_trainer(
    train_bs=BASE_TRAIN_BS,
    eval_bs=BASE_EVAL_BS,
    grad_acc=BASE_GRAD_ACC,
    lr=BASE_LR,
)

train_result = None
used_oom_fallback = False
used_lr_retry = False

try:
    train_result = trainer.train()
except RuntimeError as e:
    if "out of memory" in str(e).lower():
        print("OOM detected. Retrying with smaller batch and gradient accumulation...")
        clear_memory()
        used_oom_fallback = True
        trainer, stagnation_cb = build_trainer(
            train_bs=16,
            eval_bs=BASE_EVAL_BS,
            grad_acc=2,
            lr=BASE_LR,
        )
        train_result = trainer.train()
    else:
        raise

if stagnation_cb.triggered:
    best_ckpt = trainer.state.best_model_checkpoint
    retry_lr = max(trainer.args.learning_rate * 0.5, 1e-7)
    print(f"Auto-LR fallback triggered due to {stagnation_cb.reason}. Retrying once with LR={retry_lr}")
    used_lr_retry = True

    trainer, _ = build_trainer(
        train_bs=trainer.args.per_device_train_batch_size,
        eval_bs=trainer.args.per_device_eval_batch_size,
        grad_acc=trainer.args.gradient_accumulation_steps,
        lr=retry_lr,
    )

    if best_ckpt:
        train_result = trainer.train(resume_from_checkpoint=best_ckpt)
    else:
        train_result = trainer.train()

print("Training done.")
print("OOM fallback used:", used_oom_fallback)
print("Auto LR retry used:", used_lr_retry)
print("Best checkpoint:", trainer.state.best_model_checkpoint)

trainer.save_model(str(MODELS_DIR))
tokenizer.save_pretrained(str(MODELS_DIR))
print("Saved best model to:", MODELS_DIR)


Training mode: FULL | train samples: 95078 | valid samples: 11885 | epochs: 25 | max_steps: -1


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted
1,0.826184,0.785517,0.717122,0.718076,0.718084
2,0.673759,0.680287,0.766933,0.767606,0.767611
3,0.610158,0.715006,0.786622,0.783693,0.783697
4,0.488371,0.749355,0.811864,0.809175,0.809177
5,0.478361,0.655487,0.816155,0.816210,0.816212
6,0.376151,0.930704,0.820614,0.816888,0.816891
7,0.411962,0.917522,0.828187,0.821883,0.821885
8,0.306937,0.961470,0.830206,0.827832,0.827834
9,0.363881,0.946444,0.834329,0.828934,0.828935
10,0.308544,0.957520,0.844510,0.842840,0.842843


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted
1,0.826184,0.785517,0.717122,0.718076,0.718084
2,0.673759,0.680287,0.766933,0.767606,0.767611
3,0.610158,0.715006,0.786622,0.783693,0.783697
4,0.488371,0.749355,0.811864,0.809175,0.809177
5,0.478361,0.655487,0.816155,0.816210,0.816212
6,0.376151,0.930704,0.820614,0.816888,0.816891
7,0.411962,0.917522,0.828187,0.821883,0.821885
8,0.306937,0.961470,0.830206,0.827832,0.827834
9,0.363881,0.946444,0.834329,0.828934,0.828935
10,0.308544,0.957520,0.844510,0.842840,0.842843


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
# Final test evaluation + report artifacts
pred_output = trainer.predict(test_tok)
logits = pred_output.predictions
y_pred = np.argmax(logits, axis=1)
y_true = np.array(test_df["label"].tolist())
acc = accuracy_score(y_true, y_pred)
prec_macro, rec_macro, f1_macro, _ = precision_recall_fscore_support(
    y_true, y_pred, average="macro", zero_division=0
)
prec_weighted, rec_weighted, f1_weighted, _ = precision_recall_fscore_support(
    y_true, y_pred, average="weighted", zero_division=0
)
report_dict = classification_report(
    y_true,
    y_pred,
    target_names=labels,
    output_dict=True,
    zero_division=0,
)
report_df = pd.DataFrame(report_dict).transpose()
report_df.to_csv(REPORTS_DIR / "classification_report.csv", index=True)
cm = confusion_matrix(y_true, y_pred)
cm_df = pd.DataFrame(cm, index=labels, columns=labels)
cm_df.to_csv(REPORTS_DIR / "confusion_matrix.csv", index=True)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_df, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix (Test)")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.tight_layout()
plt.savefig(REPORTS_DIR / "confusion_matrix.png", dpi=180)
plt.close()
metrics = {
    "accuracy": float(acc),
    "precision_macro": float(prec_macro),
    "recall_macro": float(rec_macro),
    "f1_macro": float(f1_macro),
    "precision_weighted": float(prec_weighted),
    "recall_weighted": float(rec_weighted),
    "f1_weighted": float(f1_weighted),
    "best_checkpoint": trainer.state.best_model_checkpoint,
    "epochs_trained": float(trainer.state.epoch or 0),
}
with open(REPORTS_DIR / "metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("Test metrics:")
print(json.dumps(metrics, indent=2))
print("Saved:")
print("-", REPORTS_DIR / "metrics.json")
print("-", REPORTS_DIR / "classification_report.csv")
print("-", REPORTS_DIR / "confusion_matrix.csv")
print("-", REPORTS_DIR / "confusion_matrix.png")


In [ ]:
# Training curves + artifact validation checks
log_df = pd.DataFrame(trainer.state.log_history)

def plot_curve(x_col, y_col, title, out_file):
    if x_col in log_df.columns and y_col in log_df.columns:
        plt.figure(figsize=(8, 5))
        d = log_df[[x_col, y_col]].dropna()
        if len(d) > 0:
            plt.plot(d[x_col], d[y_col], marker="o")
            plt.title(title)
            plt.xlabel(x_col)
            plt.ylabel(y_col)
            plt.grid(alpha=0.3)
            plt.tight_layout()
            plt.savefig(out_file, dpi=180)
            plt.close()

curve_path = REPORTS_DIR / "training_curves.png"
if "epoch" in log_df.columns:
    plt.figure(figsize=(10, 5))
    has_any = False

    if "loss" in log_df.columns:
        d1 = log_df[["epoch", "loss"]].dropna()
        if len(d1) > 0:
            plt.plot(d1["epoch"], d1["loss"], marker="o", label="train_loss")
            has_any = True

    if "eval_loss" in log_df.columns:
        d2 = log_df[["epoch", "eval_loss"]].dropna()
        if len(d2) > 0:
            plt.plot(d2["epoch"], d2["eval_loss"], marker="o", label="eval_loss")
            has_any = True

    if "eval_f1_macro" in log_df.columns:
        d3 = log_df[["epoch", "eval_f1_macro"]].dropna()
        if len(d3) > 0:
            plt.plot(d3["epoch"], d3["eval_f1_macro"], marker="o", label="eval_f1_macro")
            has_any = True

    if has_any:
        plt.title("Training Curves")
        plt.xlabel("epoch")
        plt.legend()
        plt.grid(alpha=0.3)
        plt.tight_layout()
        plt.savefig(curve_path, dpi=180)
    plt.close()

required_files = [
    DATA_INTERIM_DIR / "merged_raw.csv",
    DATA_PROCESSED_DIR / "train.csv",
    DATA_PROCESSED_DIR / "valid.csv",
    DATA_PROCESSED_DIR / "test.csv",
    ARTIFACTS_DIR / "label_mapping.csv",
    ARTIFACTS_DIR / "class_distribution.csv",
    REPORTS_DIR / "metrics.json",
    REPORTS_DIR / "classification_report.csv",
    REPORTS_DIR / "confusion_matrix.csv",
    REPORTS_DIR / "confusion_matrix.png",
    REPORTS_DIR / "training_curves.png",
]

for p in required_files:
    assert p.exists(), f"Missing artifact: {p}"
    assert p.stat().st_size > 0, f"Empty artifact: {p}"

cm_loaded = pd.read_csv(REPORTS_DIR / "confusion_matrix.csv", index_col=0)
assert cm_loaded.shape == (6, 6), f"Unexpected confusion matrix shape: {cm_loaded.shape}"

with open(REPORTS_DIR / "metrics.json", "r") as f:
    m = json.load(f)
assert "f1_macro" in m, "f1_macro missing in metrics.json"

print("All artifact checks passed.")
print("Training curves saved to:", curve_path)


In [ ]:
from transformers import pipeline

clf = pipeline("text-classification", model=MODELS_DIR, tokenizer=tokenizer, return_all_scores=True)

def predict_emotion(text):
    # When return_all_scores=True and a single text is provided,
    # the pipeline returns a list of dictionaries, e.g.,
    # [{'label': 'Joy', 'score': 0.9}, {'label': 'Sadness', 'score': 0.1}, ...]
    # The original code used clf(text)[0], which would extract the first dictionary
    # from this list, leading to the TypeError when sorting its keys.
    # We need the entire list of scores to sort them.
    scores = clf(text)

    # Ensure 'scores' is a list of dictionaries. If the pipeline unexpectedly returns
    # List[List[Dict]] for a single input, we extract the inner list.
    if len(scores) == 1 and isinstance(scores[0], list):
        scores = scores[0]
    elif not (isinstance(scores, list) and all(isinstance(item, dict) for item in scores)):
        raise ValueError(f"Unexpected output format from pipeline: {scores}")

    scores = sorted(scores, key=lambda x: x["score"], reverse=True)
    top = scores[0]
    return top["label"], float(top["score"]), scores[:3]

predict_emotion("I failed my exam and feel hopeless")